<a href="https://colab.research.google.com/github/Kaique-Passos/FireDetection/blob/main/Comparacao_Modelos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Comparação entre os modelos

Lê os resultados dos treinos K-fold salvos no Google Drive (`MyDrive/historico_treinamentos/<modelo>/<data_hora>/`), confere se todos os treinos usaram os mesmos folds e compara as métricas.

Não precisa de GPU: só lê arquivos CSV.

In [ ]:
import os

if os.path.isdir(os.path.join('/content/FireDetection', '.git')):
  %cd /content/FireDetection
  !git pull
  print('Repositório atualizado')
else:
  !git clone https://github.com/Kaique-Passos/FireDetection.git
  print('Repositório clonado')

os.chdir('/content/FireDetection')

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive
from utils.kfold import list_runs, load_kfold_results, compare_kfold
from utils.kfold import check_same_folds, check_same_test_images

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

history_root = '/content/drive/MyDrive/historico_treinamentos'
print('Lendo treinos de:', history_root)

## Treinos disponíveis

Lista todos os treinos concluídos de cada modelo, com a configuração usada.

In [ ]:
treinos = []

for modelo in sorted(os.listdir(history_root)):
    for run in list_runs(history_root, modelo):
        with open(os.path.join(run, 'config.json')) as f:
            config = json.load(f)
        config['pasta'] = os.path.basename(run)
        treinos.append(config)

treinos = pd.DataFrame(treinos)
display(treinos)

## Escolha dos treinos

Por padrão usa o treino concluído mais recente de cada modelo. Para usar um treino específico, coloque o nome da pasta (por exemplo `'2026-10-18_21-40-12'`) em `pastas_escolhidas`.

In [ ]:
# modelos a comparar (chaves de utils/models.py); pode ter 2 ou mais
# '*_imagenet' = com pesos da ImageNet; sem o sufixo = treinado do zero
modelos = ['kutralnet_mobileoct', 'mobilenetv3_imagenet', 'shufflenetv2_imagenet']

# None = treino mais recente; ou o nome da pasta do treino
pastas_escolhidas = {
    'kutralnet_mobileoct': None,
    'mobilenetv3_imagenet': None,
    'shufflenetv2_imagenet': None
}

runs = {}
for modelo in modelos:
    pasta = pastas_escolhidas.get(modelo)
    if pasta is None:
        disponiveis = list_runs(history_root, modelo)
        if not disponiveis:
            raise ValueError('Nenhum treino concluído de {}'.format(modelo))
        runs[modelo] = disponiveis[-1]
    else:
        runs[modelo] = os.path.join(history_root, modelo, pasta)

for modelo, run in runs.items():
    print('{}: {}'.format(modelo, run))

resultados = {modelo: load_kfold_results(run) for modelo, run in runs.items()}

## Verificação dos folds e do conjunto de teste

Confere se todos os treinos usaram exatamente as mesmas imagens:
- `folds.csv`: qual imagem ficou em qual fold de validação. Iguais significa que os modelos foram treinados e validados nas mesmas partições.
- `test_images.csv`: as imagens do conjunto de teste independente (FireNet Test, 871 imagens), usado para avaliar o melhor modelo de cada fold.

O hash SHA-256 serve como comprovação compacta: hashes iguais significam arquivos idênticos.

In [ ]:
mesmos_folds, relatorio_folds = check_same_folds(list(runs.values()))
print('Folds de validação:')
display(relatorio_folds)

mesmo_teste, relatorio_teste = check_same_test_images(list(runs.values()))
print('Conjunto de teste:')
display(relatorio_teste)

if not mesmos_folds:
    raise ValueError('Os treinos NÃO usaram os mesmos folds: a comparação pareada não é válida.')
if not mesmo_teste:
    raise ValueError('Os treinos NÃO usaram as mesmas imagens de teste (ou algum treino é antigo '
                     'e não salvou o test_images.csv).')
print('Todos os treinos usaram os mesmos folds e as mesmas imagens de teste.')

## Comparação das métricas

Primeiro, a média ± desvio padrão de cada métrica nos 10 folds, com todos os modelos lado a lado. Depois, a comparação de cada par de modelos. Como os modelos usaram os mesmos folds, as amostras são pareadas: `t_test_p` é o teste t pareado e `wilcoxon_p` é o teste de Wilcoxon (não paramétrico). Um p < 0,05 indica diferença significativa.

Com 3 modelos são 3 comparações de pares, então a chance de achar alguma diferença "significativa" por acaso aumenta. Para um resultado mais rigoroso, ajuste os p-valores (por exemplo, Bonferroni: multiplique cada p por 3) ou use antes um teste para todos os modelos juntos (Friedman).

Tudo é salvo no Drive em `historico_treinamentos/comparacoes/`.

In [ ]:
from itertools import combinations

comparacao_path = os.path.join(history_root, 'comparacoes',
                               '_vs_'.join(modelos),
                               '__'.join(os.path.basename(runs[m]) for m in modelos))
os.makedirs(comparacao_path, exist_ok=True)

# média ± desvio padrão de cada métrica, todos os modelos lado a lado
metricas_resumo = [c for c in resultados[modelos[0]].columns
                   if (c.startswith('val_') or c.startswith('test_'))
                   and c.split('_', 1)[1] not in ('tp', 'fp', 'tn', 'fn')]
resumo = pd.DataFrame({
    modelo: ['{:.4f} ± {:.4f}'.format(res[m].mean(), res[m].std()) for m in metricas_resumo]
    for modelo, res in resultados.items()
}, index=metricas_resumo)
print('Média ± desvio padrão nos folds:')
display(resumo)
resumo.to_csv(os.path.join(comparacao_path, 'resumo.csv'))

# comparação pareada de cada par de modelos
for modelo_a, modelo_b in combinations(modelos, 2):
    comparacao = compare_kfold(resultados[modelo_a], resultados[modelo_b],
                               name_a=modelo_a, name_b=modelo_b)
    print('\n{} vs {}:'.format(modelo_a, modelo_b))
    display(comparacao)
    comparacao.to_csv(os.path.join(comparacao_path,
                                   'comparacao_{}_vs_{}.csv'.format(modelo_a, modelo_b)),
                      index=False)

relatorio_folds.to_csv(os.path.join(comparacao_path, 'verificacao_folds.csv'), index=False)
relatorio_teste.to_csv(os.path.join(comparacao_path, 'verificacao_teste.csv'), index=False)
print('Comparação salva em', comparacao_path)

In [ ]:
# boxplot da acurácia por fold (validação e teste), um box por modelo
metricas = [m for m in ['val_accuracy', 'test_accuracy']
            if all(m in res.columns for res in resultados.values())]
largura = max(6, 2.5 * len(modelos)) # cada gráfico cresce com o número de modelos
fig, axes = plt.subplots(1, len(metricas), figsize=(largura * len(metricas), 4.5), squeeze=False)

for ax, m in zip(axes[0], metricas):
    ax.boxplot([resultados[modelo][m] for modelo in modelos])
    ax.set_xticks(range(1, len(modelos) + 1), modelos, rotation=15)
    ax.set_title(m)
    ax.set_ylabel('Acurácia')

plt.tight_layout()
plt.savefig(os.path.join(comparacao_path, 'boxplot_acuracia.png'))
plt.show()

## Análise estatística

Espaço para a análise pedida pelo orientador.